# Fase 5 — Avaliação
Os cinco critérios de sucesso da Fase 1, cobrados um por vez sobre o K-Means com k = 5 (conjunto final de features, seção 14 do notebook 03).

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_samples

SEED = 42
RAIZ = Path('..')
PROCESSADOS = RAIZ / 'dados' / 'processados'
ESTADO = RAIZ / 'estado'
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

X_modelo = pd.read_parquet(PROCESSADOS / 'X_modelo.parquet')
base = pd.read_parquet(PROCESSADOS / 'base_lojas_clusters.parquet')
modelo_k5 = joblib.load(RAIZ / 'modelos' / 'kmeans_k5.joblib')
FEATURES_MODELO = X_modelo.columns.tolist()

## 1. Estabilidade

In [2]:
referencia = base['cluster']
K = 5

# rodar de novo com a mesma semente e a mesma base muda algo?
repeticoes_iguais = [adjusted_rand_score(referencia, KMeans(n_clusters=K, n_init=20, random_state=SEED).fit_predict(X_modelo)) for _ in range(5)]
print('ARI em 5 reexecuções com a mesma semente e a mesma base:', repeticoes_iguais)

ARI em 5 reexecuções com a mesma semente e a mesma base: [1.0, 1.0, 1.0, 1.0, 1.0]


In [3]:
def alinhar(rotulos_ref, rotulos_novos, k=K):
    # renomeia os clusters novos para os números da referência (o K-Means numera em ordem arbitrária)
    cruzado = pd.crosstab(rotulos_ref, rotulos_novos).reindex(index=range(k), columns=range(k), fill_value=0)
    linhas, colunas = linear_sum_assignment(-cruzado.values)
    return pd.Series(rotulos_novos).map(dict(zip(colunas, linhas))).values

N_REP = 30
ari = {'semente': [], 'reordenacao': [], 'subamostra80': []}
trocas = {'semente': [], 'reordenacao': [], 'subamostra80': []}
permanencia = pd.DataFrame(0, index=X_modelo.index, columns=['presente', 'mesmo_cluster'])
inercias_semente = []
for i in range(N_REP):
    m = KMeans(n_clusters=K, n_init=20, random_state=1000 + i).fit(X_modelo)
    inercias_semente.append(m.inertia_)
    ari['semente'].append(adjusted_rand_score(referencia, m.labels_))
    trocas['semente'].append((alinhar(referencia.values, m.labels_) != referencia.values).mean())

    emb = X_modelo.sample(frac=1, random_state=2000 + i)
    rot = pd.Series(KMeans(n_clusters=K, n_init=20, random_state=SEED).fit_predict(emb), index=emb.index).loc[referencia.index]
    ari['reordenacao'].append(adjusted_rand_score(referencia, rot))
    trocas['reordenacao'].append((alinhar(referencia.values, rot.values) != referencia.values).mean())

for i in range(100):
    sub = X_modelo.sample(frac=0.8, random_state=3000 + i)
    rot = KMeans(n_clusters=K, n_init=20, random_state=SEED).fit_predict(sub)
    ref_sub = referencia.loc[sub.index]
    alinhado = alinhar(ref_sub.values, rot)
    if i < N_REP:
        ari['subamostra80'].append(adjusted_rand_score(ref_sub, rot))
        trocas['subamostra80'].append((alinhado != ref_sub.values).mean())
    permanencia.loc[sub.index, 'presente'] += 1
    permanencia.loc[sub.index, 'mesmo_cluster'] += (alinhado == ref_sub.values).astype(int)

resumo_estab = pd.DataFrame({
    'ari_medio': {t: np.mean(v) for t, v in ari.items()},
    'ari_minimo': {t: np.min(v) for t, v in ari.items()},
    'execucoes_com_ari_abaixo_0.80': {t: int((np.array(v) < 0.80).sum()) for t, v in ari.items()},
    'pct_lojas_que_trocam_medio': {t: np.mean(v) * 100 for t, v in trocas.items()},
    'pct_lojas_que_trocam_maximo': {t: np.max(v) * 100 for t, v in trocas.items()},
})
print(resumo_estab.round(3).to_string())

              ari_medio  ari_minimo  execucoes_com_ari_abaixo_0.80  pct_lojas_que_trocam_medio  pct_lojas_que_trocam_maximo
semente           0.992       0.978                              0                       0.398                        0.987
reordenacao       0.990       0.968                              0                       0.490                        1.614
subamostra80      0.888       0.690                              5                       5.448                       20.628


In [4]:
# as execuções instáveis são ruído ou uma segunda solução concorrente?
print('ARI por semente (ordenado):', np.sort(np.round(ari['semente'], 3)))
print('inércias distintas encontradas:', sorted(set(np.round(inercias_semente, 1))))
print('ARI por subamostra (ordenado):', np.sort(np.round(ari['subamostra80'], 3)))

ARI por semente (ordenado): [0.978 0.982 0.984 0.985 0.988 0.989 0.989 0.989 0.989 0.989 0.991 0.991
 0.992 0.992 0.992 0.992 0.993 0.994 0.994 0.994 0.994 0.995 0.995 0.995
 0.996 0.996 0.997 0.999 1.    1.   ]
inércias distintas encontradas: [np.float64(3632.5), np.float64(3632.6), np.float64(3632.7), np.float64(3632.8)]
ARI por subamostra (ordenado): [0.69  0.749 0.75  0.772 0.773 0.802 0.815 0.82  0.824 0.832 0.86  0.901
 0.911 0.92  0.925 0.925 0.936 0.936 0.939 0.939 0.94  0.948 0.953 0.953
 0.958 0.959 0.97  0.979 0.983 0.987]


In [5]:
# estabilidade por loja: em quantas das subamostras em que apareceu ela ficou no mesmo cluster
permanencia['taxa'] = permanencia['mesmo_cluster'] / permanencia['presente']
permanencia['silhueta'] = silhouette_samples(X_modelo, referencia)
permanencia['cluster'] = referencia
faixas = pd.cut(permanencia['taxa'], [-0.01, 0.5, 0.8, 0.95, 1.0], labels=['até 50%', '50-80%', '80-95%', 'acima de 95%'])
print('lojas por taxa de permanência (100 subamostras de 80%):')
print(faixas.value_counts().sort_index().to_frame('lojas').assign(pct=lambda d: (d['lojas'] / len(permanencia) * 100).round(1)).to_string())
por_cluster = permanencia.groupby('cluster').agg(lojas=('taxa', 'size'), permanencia_media=('taxa', 'mean'),
                                                 pct_lojas_abaixo_80=('taxa', lambda s: (s < 0.8).mean() * 100),
                                                 silhueta_media=('silhueta', 'mean'))
print('\npor cluster:'); print(por_cluster.round(3).to_string())
print('\ncorrelação de Spearman entre silhueta da loja e taxa de permanência:',
      round(permanencia['silhueta'].corr(permanencia['taxa'], method='spearman'), 3))
print('silhueta média - lojas estáveis (>95%):', round(permanencia.loc[permanencia['taxa'] > 0.95, 'silhueta'].mean(), 3),
      '| instáveis (<80%):', round(permanencia.loc[permanencia['taxa'] < 0.8, 'silhueta'].mean(), 3))

lojas por taxa de permanência (100 subamostras de 80%):
              lojas   pct
taxa                     
até 50%           5   0.4
50-80%          119  10.7
80-95%          130  11.7
acima de 95%    861  77.2

por cluster:
         lojas  permanencia_media  pct_lojas_abaixo_80  silhueta_media
cluster                                                               
0          292              0.920               18.836           0.165
1          414              0.981                2.657           0.278
2           65              0.934                9.231           0.079
3          165              0.930               20.000           0.198
4          179              0.947                9.497           0.131

correlação de Spearman entre silhueta da loja e taxa de permanência: 0.683
silhueta média - lojas estáveis (>95%): 0.245 | instáveis (<80%): 0.033


In [6]:
permanencia.to_pickle(ESTADO / 'permanencia_lojas.pkl')

In [7]:
# o modelo de referência (semente 42) é a melhor solução que o K-Means encontra nesta base?
print(f'inércia do modelo de referência (semente 42, n_init=20): {modelo_k5.inertia_:.1f}')
modelo_exaustivo = KMeans(n_clusters=K, n_init=500, random_state=SEED).fit(X_modelo)
rot_exaustivo = alinhar(referencia.values, modelo_exaustivo.labels_)
print(f'inércia com n_init=500: {modelo_exaustivo.inertia_:.1f} ({(modelo_k5.inertia_ / modelo_exaustivo.inertia_ - 1) * 100:.2f}% menor que a referência)')
print('ARI referência x n_init=500:', round(adjusted_rand_score(referencia, modelo_exaustivo.labels_), 3),
      '| lojas em cluster diferente:', int((rot_exaustivo != referencia.values).sum()))
print('tamanhos referência :', np.bincount(referencia).tolist())
print('tamanhos n_init=500 :', np.bincount(rot_exaustivo).tolist())

# estabilidade medida contra a solução de menor inércia
ari_s, ari_sub = [], []
for i in range(30):
    ari_s.append(adjusted_rand_score(modelo_exaustivo.labels_, KMeans(n_clusters=K, n_init=20, random_state=1000 + i).fit_predict(X_modelo)))
    sub = X_modelo.sample(frac=0.8, random_state=3000 + i)
    ari_sub.append(adjusted_rand_score(pd.Series(modelo_exaustivo.labels_, index=X_modelo.index).loc[sub.index],
                                       KMeans(n_clusters=K, n_init=20, random_state=SEED).fit_predict(sub)))
print(f'\ncontra a solução de menor inércia: ARI semente médio {np.mean(ari_s):.3f} (mín {np.min(ari_s):.3f}) '
      f'| ARI subamostra80 médio {np.mean(ari_sub):.3f} (mín {np.min(ari_sub):.3f})')

inércia do modelo de referência (semente 42, n_init=20): 3632.6


inércia com n_init=500: 3632.5 (0.00% menor que a referência)
ARI referência x n_init=500: 0.997 | lojas em cluster diferente: 2
tamanhos referência : [292, 414, 65, 165, 179]
tamanhos n_init=500 : [293, 414, 63, 165, 180]



contra a solução de menor inércia: ARI semente médio 0.991 (mín 0.981) | ARI subamostra80 médio 0.888 (mín 0.691)


## 2. Os clusters são melhores que o cadastro?
### Teste A — concordância com tipo_loja e sortimento

In [8]:
concordancia = pd.DataFrame({
    'ARI': {c: adjusted_rand_score(base[c], referencia) for c in ['tipo_loja', 'sortimento']},
    'NMI': {c: normalized_mutual_info_score(base[c], referencia) for c in ['tipo_loja', 'sortimento']},
})
print(concordancia.round(3).to_string())
print('\ncluster x tipo_loja (% de cada tipo em cada cluster):')
print((pd.crosstab(base['tipo_loja'], referencia, normalize='index') * 100).round(1).to_string())
print('\ncluster x sortimento (% de cada sortimento em cada cluster):')
print((pd.crosstab(base['sortimento'], referencia, normalize='index') * 100).round(1).to_string())

              ARI    NMI
tipo_loja   0.118  0.148
sortimento  0.056  0.065

cluster x tipo_loja (% de cada tipo em cada cluster):
cluster       0     1     2     3     4
tipo_loja                              
a          33.1  18.8   6.3  21.1  20.8
b          11.8   0.0  11.8   0.0  76.5
c          34.5  24.3  14.9   6.1  20.3
d          11.5  76.1   0.9   8.3   3.2

cluster x sortimento (% de cada sortimento em cada cluster):
cluster        0     1     2     3     4
sortimento                              
a           33.7  25.0   2.9  20.6  17.9
b            0.0   0.0  22.2   0.0  77.8
c           17.9  51.9   9.0   8.4  12.9


### Teste B — variância explicada (eta²)
eta² = variação entre as médias dos grupos / variação total. É o R² de explicar a métrica só com o grupo da loja.

In [9]:
def eta2(valores, grupos):
    media_geral = valores.mean()
    entre = valores.groupby(grupos).agg(lambda s: len(s) * (s.mean() - media_geral) ** 2).sum()
    return entre / ((valores - media_geral) ** 2).sum()

metricas_negocio = {
    'venda_media_dia': 'no modelo', 'ticket_medio': 'no modelo', 'efeito_promo': 'no modelo',
    'volatilidade': 'no modelo', 'sazonalidade': 'no modelo', 'distancia_concorrente': 'no modelo',
    'clientes_medios_dia': 'fora do modelo', 'pico_sabado': 'fora do modelo', 'efeito_promo_dias_uteis': 'fora do modelo',
    'prop_domingos_abertos': 'fora do modelo',
}
gerador = np.random.default_rng(SEED)
linhas = []
for metrica, origem in metricas_negocio.items():
    aleatorio = np.mean([eta2(base[metrica], pd.Series(gerador.permutation(referencia.values), index=base.index)) for _ in range(200)])
    linhas.append({'metrica': metrica, 'origem': origem,
                   'tipo_loja': eta2(base[metrica], base['tipo_loja']) * 100,
                   'sortimento': eta2(base[metrica], base['sortimento']) * 100,
                   'cluster': eta2(base[metrica], referencia) * 100,
                   'cluster_aleatorio': aleatorio * 100})
tabela_eta = pd.DataFrame(linhas).set_index('metrica')
tabela_eta['ganho_pp'] = tabela_eta['cluster'] - tabela_eta['tipo_loja']
tabela_eta['vezes'] = tabela_eta['cluster'] / tabela_eta['tipo_loja']
print(tabela_eta.round(1).to_string())
print('\nmédia - no modelo  : tipo_loja', round(tabela_eta[tabela_eta['origem'].str.startswith('no')]['tipo_loja'].mean(), 1),
      '| cluster', round(tabela_eta[tabela_eta['origem'].str.startswith('no')]['cluster'].mean(), 1))
print('média - fora do modelo: tipo_loja', round(tabela_eta[tabela_eta['origem'] == 'fora do modelo']['tipo_loja'].mean(), 1),
      '| cluster', round(tabela_eta[tabela_eta['origem'] == 'fora do modelo']['cluster'].mean(), 1))

                                 origem  tipo_loja  sortimento  cluster  cluster_aleatorio  ganho_pp  vezes
metrica                                                                                                    
venda_media_dia               no modelo        2.8         2.5     28.3                0.3      25.5   10.2
ticket_medio                  no modelo       42.3        11.1     50.6                0.4       8.3    1.2
efeito_promo                  no modelo        8.4         9.3     57.8                0.4      49.4    6.9
volatilidade                  no modelo        2.0         1.5     60.8                0.4      58.8   30.2
sazonalidade                  no modelo        3.1        10.1     43.2                0.3      40.1   14.0
distancia_concorrente         no modelo        2.5         2.4     20.7                0.3      18.2    8.2
clientes_medios_dia      fora do modelo       25.5        10.9     36.7                0.4      11.2    1.4
pico_sabado              for

In [10]:
# comparação com o mesmo número de grupos do tipo_loja (k = 4)
rot_k4 = pd.Series(KMeans(n_clusters=4, n_init=20, random_state=SEED).fit_predict(X_modelo), index=base.index)
print(pd.DataFrame({'tipo_loja (4 grupos)': {m: eta2(base[m], base['tipo_loja']) * 100 for m in metricas_negocio},
                    'cluster k=4': {m: eta2(base[m], rot_k4) * 100 for m in metricas_negocio}}).round(1).to_string())

                         tipo_loja (4 grupos)  cluster k=4
venda_media_dia                           2.8         15.9
ticket_medio                             42.3         49.7
efeito_promo                              8.4         51.6
volatilidade                              2.0         60.8
sazonalidade                              3.1         16.4
distancia_concorrente                     2.5         35.0
clientes_medios_dia                      25.5         28.2
pico_sabado                              22.4         41.0
efeito_promo_dias_uteis                   6.0         25.3
prop_domingos_abertos                    70.0          4.1


## 3. Perfil médio de cada cluster

In [11]:
perfil = base.groupby('cluster').agg(
    lojas=('cluster', 'size'),
    venda_media_dia=('venda_media_dia', 'mean'),
    clientes_dia=('clientes_medios_dia', 'mean'),
    ticket_medio=('ticket_medio', 'mean'),
    efeito_promo=('efeito_promo', 'mean'),
    volatilidade=('volatilidade', 'mean'),
    sazonalidade=('sazonalidade', 'mean'),
    pico_sabado=('pico_sabado', 'mean'),
    efeito_promo_dias_uteis=('efeito_promo_dias_uteis', 'mean'),
    distancia_mediana=('distancia_concorrente', 'median'),
    promo_continua=('promo_continua', 'mean'),
    prop_domingos_abertos=('prop_domingos_abertos', 'mean'),
    lojas_abrem_domingo=('abre_domingo', 'sum'),
)
rede = base.agg({'venda_media_dia': 'mean', 'clientes_medios_dia': 'mean', 'ticket_medio': 'mean', 'efeito_promo': 'mean',
                 'volatilidade': 'mean', 'sazonalidade': 'mean', 'pico_sabado': 'mean', 'efeito_promo_dias_uteis': 'mean', 'distancia_concorrente': 'median',
                 'promo_continua': 'mean', 'prop_domingos_abertos': 'mean'})
perfil_t = perfil.T
perfil_t.columns = [f'cluster {c}' for c in perfil_t.columns]
perfil_t['rede'] = [len(base), rede['venda_media_dia'], rede['clientes_medios_dia'], rede['ticket_medio'], rede['efeito_promo'],
                    rede['volatilidade'], rede['sazonalidade'], rede['pico_sabado'], rede['efeito_promo_dias_uteis'], rede['distancia_concorrente'],
                    rede['promo_continua'], rede['prop_domingos_abertos'], base['abre_domingo'].sum()]
print(perfil_t.round(3).to_string())

                         cluster 0  cluster 1  cluster 2  cluster 3  cluster 4      rede
lojas                      292.000    414.000     65.000    165.000    179.000  1115.000
venda_media_dia           5718.793   7084.153   6312.058   6124.852   9544.712  6934.632
clientes_dia               692.628    634.199    793.281    629.902   1234.847   754.565
ticket_medio                 8.435     11.353      8.245      9.782      8.042     9.644
efeito_promo                 0.454      0.349      0.320      0.708      0.263     0.414
volatilidade                 0.275      0.251      0.312      0.364      0.213     0.272
sazonalidade                 0.084      0.092      0.159      0.094      0.076     0.092
pico_sabado                  0.750      0.921      0.891      0.524      0.931     0.817
efeito_promo_dias_uteis      0.416      0.393      0.318      0.529      0.297     0.399
distancia_mediana          840.000   5835.000   2640.000   4150.000    770.000  2325.000
promo_continua       

In [12]:
perfil.to_pickle(ESTADO / 'perfil_clusters.pkl')
tabela_eta.to_pickle(ESTADO / 'tabela_eta.pkl')

## 4. Onde caíram as 17 lojas do tipo b

In [2]:
tipo_b = base[base['tipo_loja'] == 'b'].copy()
distribuicao_b = tipo_b['cluster'].value_counts().rename('lojas_tipo_b').to_frame()
distribuicao_b['pct_do_tipo_b'] = (distribuicao_b['lojas_tipo_b'] / len(tipo_b) * 100).round(1)
distribuicao_b['lojas_no_cluster'] = base['cluster'].value_counts()
distribuicao_b['pct_do_cluster'] = (distribuicao_b['lojas_tipo_b'] / distribuicao_b['lojas_no_cluster'] * 100).round(1)
print(distribuicao_b.to_string())

# chance de 13 ou mais das 17 caírem no cluster 4 por acaso (hipergeométrica)
from scipy.stats import hypergeom
n_c4 = int((base['cluster'] == 4).sum())
print(f'\nesperado no cluster 4 por acaso: {17 * n_c4 / len(base):.1f} lojas | observado: 13 | p = {hypergeom.sf(12, len(base), n_c4, 17):.1e}')

         lojas_tipo_b  pct_do_tipo_b  lojas_no_cluster  pct_do_cluster
cluster                                                               
4                  13           76.5               179             7.3
2                   2           11.8                65             3.1
0                   2           11.8               292             0.7

esperado no cluster 4 por acaso: 2.7 lojas | observado: 13 | p = 4.2e-08


In [3]:
colunas_perfil = ['venda_media_dia', 'clientes_medios_dia', 'ticket_medio', 'efeito_promo', 'volatilidade', 'sazonalidade',
                  'distancia_concorrente', 'prop_domingos_abertos']
perfil_b = tipo_b.groupby('cluster')[colunas_perfil].mean()
perfil_b.insert(0, 'lojas', tipo_b.groupby('cluster').size())
perfil_b.loc['tipo b (17)'] = [17] + tipo_b[colunas_perfil].mean().tolist()
perfil_b.loc['cluster 4 inteiro'] = [n_c4] + base.loc[base['cluster'] == 4, colunas_perfil].mean().tolist()
perfil_b.loc['rede'] = [len(base)] + base[colunas_perfil].mean().tolist()
print(perfil_b.T.round(3).to_string())

cluster                       0         2          4  tipo b (17)  cluster 4 inteiro      rede
lojas                     2.000     2.000     13.000       17.000            179.000  1115.000
venda_media_dia        9619.429  5879.146  10839.397    10112.313           9544.712  6934.632
clientes_medios_dia    1661.230  1340.168   2152.358     1999.026           1234.847   754.565
ticket_medio              5.952     4.328      5.177        5.168              8.042     9.644
efeito_promo              0.689    -0.007      0.145        0.191              0.263     0.414
volatilidade              0.361     0.328      0.219        0.248              0.213     0.272
sazonalidade              0.074     0.111      0.070        0.076              0.076     0.092
distancia_concorrente   245.000  2525.000    960.769     1060.588           1312.123  5396.614
prop_domingos_abertos     0.903     0.985      0.996        0.984              0.072     0.024


In [4]:
# o que o modelo enxergou: posição das 17 lojas nas 6 features (desvios-padrão da rede)
z_b = X_modelo.loc[tipo_b.index]
print('z médio das lojas do tipo b, por cluster de destino:')
print(z_b.groupby(tipo_b['cluster']).mean().round(2).T.to_string())
print('\ncentroide do cluster 4:', dict(zip(FEATURES_MODELO, modelo_k5.cluster_centers_[4].round(2))))

z médio das lojas do tipo b, por cluster de destino:
cluster                   0     2     4
venda_media_dia        0.95 -0.52  1.33
ticket_medio          -2.16 -3.56 -2.87
efeito_promo           1.47 -2.77 -1.68
volatilidade           1.52  0.98 -0.93
sazonalidade          -0.67  0.74 -0.79
distancia_concorrente -1.54  0.05 -0.62

centroide do cluster 4: {'venda_media_dia': np.float64(1.02), 'ticket_medio': np.float64(-0.84), 'efeito_promo': np.float64(-0.87), 'volatilidade': np.float64(-1.02), 'sazonalidade': np.float64(-0.6), 'distancia_concorrente': np.float64(-0.73)}


In [5]:
# loja a loja: cluster, distância ao próprio centroide e ao centroide do cluster 4, e estabilidade
permanencia = pd.read_pickle(ESTADO / 'permanencia_lojas.pkl')
distancias = pd.DataFrame(modelo_k5.transform(X_modelo), index=X_modelo.index)
detalhe_b = tipo_b[['sortimento', 'cluster', 'venda_media_dia', 'clientes_medios_dia', 'ticket_medio', 'sazonalidade']].copy()
detalhe_b['dist_proprio'] = [distancias.loc[l, c] for l, c in tipo_b['cluster'].items()]
detalhe_b['dist_cluster_4'] = distancias.loc[tipo_b.index, 4]
detalhe_b['permanencia'] = permanencia.loc[tipo_b.index, 'taxa']
print(detalhe_b.sort_values(['cluster', 'venda_media_dia']).round(3).to_string())
print('\nposição das 17 no ranking de ticket da rede (1 = menor):', sorted(base['ticket_medio'].rank().loc[tipo_b.index].astype(int).tolist()))
print('lojas do cluster 4 que não são tipo b - ticket médio:', round(base.loc[(base['cluster'] == 4) & (base['tipo_loja'] != 'b'), 'ticket_medio'].mean(), 2),
      '| clientes:', round(base.loc[(base['cluster'] == 4) & (base['tipo_loja'] != 'b'), 'clientes_medios_dia'].mean(), 0))

     sortimento  cluster  venda_media_dia  clientes_medios_dia  ticket_medio  sazonalidade  dist_proprio  dist_cluster_4  permanencia
loja                                                                                                                                 
1081          a        0         5924.919              937.178         6.322         0.086         2.189           3.722        1.000
335           a        0        13313.939             2385.282         5.582         0.061         4.261           4.415        0.892
274           b        2         4070.828              992.741         4.101         0.102         4.783           5.325        0.959
676           b        2         7687.464             1687.594         4.555         0.119         3.542           3.798        0.818
512           b        4         5510.979             1287.972         4.279         0.064         3.385           3.385        1.000
353           b        4         5651.369             1451.086

## 5. Acionável: alavancas de cada cluster e matriz de decisão

In [2]:
from scipy.spatial.distance import pdist, squareform

alavancas = ['venda_media_dia', 'ticket_medio', 'clientes_medios_dia', 'efeito_promo', 'efeito_promo_dias_uteis',
             'volatilidade', 'sazonalidade', 'pico_sabado', 'distancia_concorrente', 'prop_domingos_abertos']
# posição de cada cluster em relação à rede, em desvios-padrão (média do cluster - média da rede) / desvio da rede
z_cluster = ((base.groupby('cluster')[alavancas].mean() - base[alavancas].mean()) / base[alavancas].std()).T
print('posição de cada cluster em desvios-padrão da rede:')
print(z_cluster.round(2).to_string())

posição de cada cluster em desvios-padrão da rede:
cluster                     0     1     2     3     4
venda_media_dia         -0.51  0.06 -0.26 -0.34  1.09
ticket_medio            -0.61  0.86 -0.70  0.07 -0.81
clientes_medios_dia     -0.18 -0.34  0.11 -0.35  1.36
efeito_promo             0.22 -0.36 -0.52  1.62 -0.84
efeito_promo_dias_uteis  0.13 -0.05 -0.64  1.03 -0.81
volatilidade             0.07 -0.36  0.70  1.58 -1.00
sazonalidade            -0.28  0.03  2.47  0.08 -0.59
pico_sabado             -0.31  0.48  0.34 -1.36  0.52
distancia_concorrente   -0.53  0.34  0.55  0.44 -0.53
prop_domingos_abertos   -0.11 -0.16  0.77 -0.09  0.34


In [3]:
venda = base['venda_media_dia']
mediana_cluster = venda.groupby(base['cluster']).transform('median')
gap_pct = ((mediana_cluster - venda) / mediana_cluster).where(venda < mediana_cluster)
meta = pd.DataFrame({
    'lojas': base['cluster'].value_counts().sort_index(),
    'mediana_venda': venda.groupby(base['cluster']).median(),
    'cv_venda': venda.groupby(base['cluster']).std() / venda.groupby(base['cluster']).mean(),
    'gap_mediano_pct_abaixo': gap_pct.groupby(base['cluster']).median() * 100,
    'gap_p90_pct_abaixo': gap_pct.groupby(base['cluster']).quantile(0.9) * 100,
})
print(meta.round(3).to_string())
print('\nrede: mediana', round(venda.median(), 0), '| cv', round(venda.std() / venda.mean(), 3))

         lojas  mediana_venda  cv_venda  gap_mediano_pct_abaixo  gap_p90_pct_abaixo
cluster                                                                            
0          292       5476.370     0.269                  16.115              33.599
1          414       6959.924     0.245                  15.795              31.414
2           65       6324.801     0.268                  25.125              43.580
3          165       5909.245     0.297                  16.205              39.614
4          179       8837.105     0.341                  19.089              35.439

rede: mediana 6590.0 | cv 0.344


In [4]:
# dois clusters levam à mesma ação? distância entre centroides e a maior diferença de cada par
centros = pd.DataFrame(modelo_k5.cluster_centers_, columns=FEATURES_MODELO)
dist_centros = pd.DataFrame(squareform(pdist(centros)), index=centros.index, columns=centros.index)
pares = []
for a in range(5):
    for b in range(a + 1, 5):
        diff = (centros.loc[a] - centros.loc[b]).abs()
        pares.append({'par': f'{a}-{b}', 'distancia_centroides': dist_centros.loc[a, b], 'maior_diferenca_z': diff.max(),
                      'feature_que_mais_separa': diff.idxmax(), 'features_com_diferenca_acima_1z': int((diff > 1).sum())})
print(pd.DataFrame(pares).sort_values('distancia_centroides').round(2).to_string(index=False))

par  distancia_centroides  maior_diferenca_z feature_que_mais_separa  features_com_diferenca_acima_1z
0-1                  2.22               1.43            ticket_medio                                2
0-4                  2.27               1.58         venda_media_dia                                3
0-3                  2.39               1.48            volatilidade                                3
1-4                  2.56               1.67            ticket_medio                                2
1-3                  2.83               1.91            volatilidade                                2
0-2                  3.06               2.72            sazonalidade                                1
1-2                  3.11               2.40            sazonalidade                                3
2-3                  3.37               2.35            sazonalidade                                2
2-4                  3.86               3.04            sazonalidade              

## 6. Quantificado: simulação de recuperação parcial do gap
Premissas: referência = venda média por dia de operação; benchmark = mediana do próprio cluster;
só lojas abaixo da mediana recuperam; dias de operação no ano = proporção de dias abertos da loja x 365.

In [5]:
perfil_lojas = pd.read_pickle(ESTADO / 'perfil_lojas.pkl')
dias_ano = perfil_lojas.loc[base.index, 'pct_dias_abertos'] * 365
abaixo = venda < mediana_cluster
gap_dia = (mediana_cluster - venda).where(abaixo, 0)
gap_ano = gap_dia * dias_ano
faturamento_ano = (venda * dias_ano).sum()

print(f'lojas abaixo da mediana do cluster: {abaixo.sum()} de {len(base)}')
print(f'dias de operação por ano (média): {dias_ano.mean():.1f} | mín {dias_ano.min():.1f} | máx {dias_ano.max():.1f}')
print(f'faturamento anual estimado da rede: R$ {faturamento_ano / 1e6:,.1f} mi')
print(f'gap diário total: R$ {gap_dia.sum():,.0f} | gap anual total (100%): R$ {gap_ano.sum() / 1e6:,.1f} mi')
print(f'gap percentual das lojas abaixo: mediana {gap_pct.median() * 100:.1f}% | p90 {gap_pct.quantile(0.9) * 100:.1f}% | máx {gap_pct.max() * 100:.1f}%')

cenarios = pd.DataFrame({'captura': [0.10, 0.25, 0.50]})
cenarios['ganho_anual_R$_mi'] = cenarios['captura'] * gap_ano.sum() / 1e6
cenarios['pct_do_faturamento_da_rede'] = cenarios['captura'] * gap_ano.sum() / faturamento_ano * 100
cenarios['ganho_com_300_dias_R$_mi'] = cenarios['captura'] * gap_dia.sum() * 300 / 1e6
print(cenarios.round(2).to_string(index=False))

lojas abaixo da mediana do cluster: 556 de 1115
dias de operação por ano (média): 302.9 | mín 244.5 | máx 365.0
faturamento anual estimado da rede: R$ 2,345.7 mi
gap diário total: R$ 679,161 | gap anual total (100%): R$ 205.5 mi
gap percentual das lojas abaixo: mediana 16.8% | p90 35.0% | máx 54.2%
 captura  ganho_anual_R$_mi  pct_do_faturamento_da_rede  ganho_com_300_dias_R$_mi
    0.10              20.55                        0.88                     20.37
    0.25              51.38                        2.19                     50.94
    0.50             102.76                        4.38                    101.87


In [6]:
por_cluster = pd.DataFrame({
    'lojas_abaixo': abaixo.groupby(base['cluster']).sum(),
    'gap_anual_R$_mi': gap_ano.groupby(base['cluster']).sum() / 1e6,
    'captura_25pct_R$_mi': gap_ano.groupby(base['cluster']).sum() * 0.25 / 1e6,
    'faturamento_anual_R$_mi': (venda * dias_ano).groupby(base['cluster']).sum() / 1e6,
})
por_cluster['captura_25pct_sobre_faturamento_pct'] = por_cluster['captura_25pct_R$_mi'] / por_cluster['faturamento_anual_R$_mi'] * 100
print(por_cluster.round(2).to_string())

         lojas_abaixo  gap_anual_R$_mi  captura_25pct_R$_mi  faturamento_anual_R$_mi  captura_25pct_sobre_faturamento_pct
cluster                                                                                                                  
0                 146            41.69                10.42                   504.98                                 2.06
1                 207            74.13                18.53                   884.57                                 2.10
2                  32            14.53                 3.63                   126.32                                 2.87
3                  82            28.76                 7.19                   305.66                                 2.35
4                  89            46.40                11.60                   524.13                                 2.21


In [7]:
# sensibilidade: o mesmo cálculo com outras referências de pares
for nome, grupos in {'rede inteira': pd.Series(0, index=base.index), 'tipo_loja': base['tipo_loja'], 'cluster': base['cluster']}.items():
    med = venda.groupby(grupos).transform('median')
    g = ((med - venda).clip(lower=0) * dias_ano).sum()
    print(f'{nome:<13}: gap anual R$ {g / 1e6:6.1f} mi | captura 25% R$ {g * 0.25 / 1e6:5.1f} mi | CV da venda nos grupos {(venda.groupby(grupos).std() / venda.groupby(grupos).mean()).mean():.3f}')

rede inteira : gap anual R$  228.1 mi | captura 25% R$  57.0 mi | CV da venda nos grupos 0.344
tipo_loja    : gap anual R$  232.4 mi | captura 25% R$  58.1 mi | CV da venda nos grupos 0.351
cluster      : gap anual R$  205.5 mi | captura 25% R$  51.4 mi | CV da venda nos grupos 0.284
